In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

doc = documents[0]
user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [21]:
result = response.output_parsed

print(result)

questions=['What does RAG actually do to help an LLM answer questions better?', 'Why do LLMs need help with things like fresh info or private documents?', 'What kind of FAQ assistant are you building in this module?', 'What are the main things covered in the first part of this module?', 'How is the second part of the module different from the first one?']


In [ ]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)
usage

['What exactly is a Retrieval-Augmented Generation system, and why does it help with things like outdated knowledge or missing access to your data?', 'Why does this module build the RAG example in plain Python instead of using a framework right away?', 'How does the lesson describe a large language model in simple terms, and what are its main limits?', 'What kind of FAQ chatbot are you building in this module, and what sort of question should it be able to answer?', 'What will the first part of the module cover, and how is the second part different?']


ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=125, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1145)

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

[{'question': 'What does RAG actually do to help an LLM answer questions better?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why do LLMs need help with things like fresh info or private documents?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of FAQ assistant are you building in this module?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main things covered in the first part of this module?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'How is the second part of the module different from the first one?',
  'document': '01-agentic-rag/lessons/01-intro.md'}]

In [26]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

Q1 - Generating Ground Truth for first 3 pages only

In [27]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [28]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=121, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1141),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1393),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1853)]

The full ground truth is imported here

In [31]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [32]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

Text and Vector Search, and hybrid search

In [33]:
texts = [doc["content"] for doc in chunks]

In [39]:
from tqdm.auto import tqdm
from embedder import Embedder
import numpy as np

embed = Embedder()

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [82]:
from minsearch import Index
index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
index.fit(chunks)

def text_search(question, num=5):
    boost_dict = {"content": 2.0, "filename": 0.5}

    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=num
    )

In [83]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

def vector_search(query, num=5):
    query_vector = embed.encode(query)

    return vindex.search(query_vector, num_results=num)    

In [84]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [101]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num=10)
    vector_results = vector_search(query, num=10)
    return rrf([text_results, vector_results], k=k)

Q2 - 

In [45]:
q = ground_truth[0]["question"]

In [46]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [47]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [50]:
q

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [93]:
def compute_relevance(q, search_function, *args, **kwargs):
    doc_id = q["filename"]
    results = search_function(q["question"], *args, **kwargs)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [94]:
def compute_relevance_total(ground_truth, search_function, *args, **kwargs):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, *args, **kwargs)
        relevance_total.append(relevance)

    return relevance_total

In [67]:
q = ground_truth[0]
print(q["question"])
compute_relevance(q, text_search)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[0, 0, 0, 0, 1]

Q4 - Evaluate Text Search and Give Hit Rate

In [65]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [70]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [95]:
def evaluate(ground_truth, search_function, *args, **kwargs):
    relevance_total = compute_relevance_total(ground_truth, search_function, *args, **kwargs)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [96]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

Q5 - Evaluate vector search and find out MRR

In [97]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

Q6 - Hybrid evaluation and tuning

In [102]:
evaluate(
    ground_truth,
    hybrid_search,
    1
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}

In [107]:
evaluate(
    ground_truth,
    hybrid_search,
    50
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [104]:
evaluate(
    ground_truth,
    hybrid_search,
    100
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [105]:
evaluate(
    ground_truth,
    hybrid_search,
    200
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}